# Can a pair of smart glasses help you hear the conversation?
**Side project inspired by the ECHI-2 task (multi-microphone conversation enhancement for assistive listening). Author: Alaa Guedda.**

This single notebook (free Colab T4, about 40 minutes) does everything:

1. downloads open data (LibriSpeech speech + DEMAND 16-channel real noise),
2. simulates a 4-microphone glasses array with a wearer and two conversation partners (`pyroomacoustics`),
3. trains small mask-based GRU enhancers and answers three research questions:
   - **RQ1** Do more microphones help (1 vs 2 vs 4)?
   - **RQ2** What does low latency (causal processing) cost?
   - **RQ3** Where does it break (own-voice loudness, noise level, reverberation, noise type)?
4. writes one self-contained, page-by-page HTML presentation: `echi_proxy_presentation.html`.

Set `QUICK = True` in the config cell for a 2-minute smoke test.


In [1]:
!pip -q install pyroomacoustics soundfile pystoi pesq pandas scipy


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, glob, time, json, math, base64, io, copy, subprocess, tarfile, zipfile
import numpy as np, pandas as pd, soundfile as sf, torch, torch.nn as nn
from pathlib import Path

T_START = time.time()
QUICK = os.environ.get("ECHI_QUICK") == "1"            # True = 2-minute smoke test
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "CPU")

FS = 16000; SEG_S = 3; T = FS * SEG_S
ROOMS = dict(train=420, val=30, test=60) if not QUICK else dict(train=6, val=2, test=3)
MIX_PER_ROOM = 3
STEPS = 3000 if not QUICK else 20
BATCH, HID = 16, 256
EVAL_EVERY = 250 if not QUICK else 10
NFFT, HOP = 512, 256; NB = NFFT // 2 + 1                # 32 ms window, 16 ms hop

DATA = Path(os.environ.get("ECHI_DATA", "data")); DATA.mkdir(exist_ok=True)
DEMAND_ENVS = ["DLIVING", "PCAFETER", "STRAFFIC", "OMEETING"]

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0: return True
    r = subprocess.run(["wget", "-q", "-c", "-O", str(dest), url])
    ok = r.returncode == 0 and dest.exists() and dest.stat().st_size > 0
    print(("downloaded: " if ok else "FAILED: ") + dest.name); return ok

if not (DATA / "LibriSpeech").exists() and download("https://www.openslr.org/resources/12/dev-clean.tar.gz", DATA / "dev-clean.tar.gz"):
    with tarfile.open(DATA / "dev-clean.tar.gz") as t: t.extractall(DATA)
for env in DEMAND_ENVS:
    if not (DATA / "demand" / env).exists() and download(f"https://zenodo.org/records/1227121/files/{env}_16k.zip", DATA / f"{env}_16k.zip"):
        with zipfile.ZipFile(DATA / f"{env}_16k.zip") as z: z.extractall(DATA / "demand")

# ---- index speech (utterances >= segment length) and split SPEAKERS: train / val / test never overlap
files = sorted(glob.glob(str(DATA / "LibriSpeech/dev-clean/**/*.flac"), recursive=True))
sp = pd.DataFrame([dict(path=f, speaker=Path(f).parts[-3], dur=sf.info(f).duration) for f in files])
sp = sp[sp.dur >= SEG_S + 0.05]
spk_all = np.random.default_rng(0).permutation(sorted(sp.speaker.unique()))
n_sp = len(spk_all); n_te = max(3, n_sp // 5); n_va = max(3, n_sp // 10)
SPK = dict(test=list(spk_all[:n_te]), val=list(spk_all[n_te:n_te + n_va]), train=list(spk_all[n_te + n_va:]))
SPK_UTT = {s: g.path.tolist() for s, g in sp.groupby("speaker")}
NOISE = {e: sorted(glob.glob(str(DATA / "demand" / e / "ch*.wav"))) for e in DEMAND_ENVS if (DATA / "demand" / e).exists()}
ENVS = sorted(NOISE)
NOISE_FRAC = dict(train=(0.0, 0.67), val=(0.67, 0.83), test=(0.83, 1.0))   # noise time ranges never overlap
print("speakers train/val/test:", {k: len(v) for k, v in SPK.items()}, "| noise envs:", ENVS)


device: cuda | Tesla T4
downloaded: dev-clean.tar.gz


/tmp/ipykernel_1090/1255715469.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(DATA / "dev-clean.tar.gz") as t: t.extractall(DATA)


downloaded: DLIVING_16k.zip
downloaded: PCAFETER_16k.zip
downloaded: STRAFFIC_16k.zip
downloaded: OMEETING_16k.zip
speakers train/val/test: {'test': 8, 'val': 4, 'train': 28} | noise envs: ['DLIVING', 'OMEETING', 'PCAFETER', 'STRAFFIC']


In [3]:
import pyroomacoustics as pra
from scipy.signal import fftconvolve
from multiprocessing import Pool

# 4 mics on a glasses frame (m, head frame: x forward, y left, z up) + wearer's mouth
MICS = np.array([[0.07, 0.075, 0], [0.0, 0.075, 0], [0.07, -0.075, 0], [0.0, -0.075, 0]]).T
MOUTH = np.array([0.10, 0.0, -0.12])

def _unit(c): return c / (np.sqrt(np.mean(c[0] ** 2)) + 1e-8)       # unit RMS at the reference mic

def sim_room(job):
    seed, spks, (f0, f1) = job
    rng = np.random.default_rng(seed)
    rt60 = float(rng.uniform(0.2, 0.8)); dim = rng.uniform([4, 4, 2.5], [9, 8, 3.5])
    ea, mo = pra.inverse_sabine(rt60, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(ea), max_order=min(mo, 20))
    head = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 1.6])
    yaw = rng.uniform(0, 2 * np.pi); c, s = np.cos(yaw), np.sin(yaw)
    Rz = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    room.add_microphone_array(pra.MicrophoneArray(Rz @ MICS + head[:, None], FS))
    room.add_source(Rz @ MOUTH + head)                                  # source 0: the wearer
    for _ in range(2):                                                  # sources 1-2: partners in front, 0.8-3 m
        ang, d = rng.uniform(-1.3, 1.3), rng.uniform(0.8, 3.0)
        room.add_source(np.clip(head + Rz @ np.array([d * np.cos(ang), d * np.sin(ang), rng.uniform(-.1, .1)]), 0.3, dim - 0.3))
    room.compute_rir()
    outs, metas = [], []
    for _ in range(MIX_PER_ROOM):
        imgs = []
        for si, spk in enumerate(rng.choice(spks, 3, replace=False)):
            paths = SPK_UTT[spk]; path = paths[int(rng.integers(len(paths)))]
            st = int(rng.integers(0, sf.info(path).frames - T + 1))
            x, _ = sf.read(path, start=st, frames=T, dtype="float32")
            x = x / (np.sqrt(np.mean(x ** 2)) + 1e-8)
            if si == 2: x = x * 10 ** (rng.uniform(-3, 3) / 20)
            imgs.append(np.stack([fftconvolve(x, room.rir[m][si])[:T] for m in range(4)]))
        env = ENVS[int(rng.integers(len(ENVS)))]; chs = rng.choice(len(NOISE[env]), 4, replace=False)
        nfr = sf.info(NOISE[env][0]).frames
        st = int(rng.integers(int(f0 * nfr), int(f1 * nfr) - T))
        nz = np.stack([sf.read(NOISE[env][ch], start=st, frames=T, dtype="float32")[0] for ch in chs])
        # components at unit RMS: [partners (target), wearer's own voice, noise]; re-mixed on the fly
        outs.append(np.stack([_unit(imgs[1] + imgs[2]), _unit(imgs[0]), _unit(nz)]).astype(np.float16))
        metas.append(dict(env=env, rt60=rt60))
    return outs, metas

def build_split(name, seed0):
    t0 = time.time()
    jobs = [(seed0 + i, SPK[name], NOISE_FRAC[name]) for i in range(ROOMS[name])]
    with Pool(max(1, os.cpu_count() or 1)) as pool: res = list(pool.imap(sim_room, jobs, chunksize=2))
    comp = torch.from_numpy(np.concatenate([np.stack(r[0]) for r in res]))
    meta = pd.DataFrame([m for r in res for m in r[1]])
    print(f"{name}: {len(comp)} mixtures from {ROOMS[name]} rooms in {time.time()-t0:.0f}s | {comp.numel()*2/1e9:.2f} GB")
    return comp, meta

TR, meta_tr = build_split("train", 1000)
VA, meta_va = build_split("val", 500000)
TE, meta_te = build_split("test", 900000)

def draw_levels(n, seed):      # test/val: wearer 12-26 dB above partners (pilot study range), SNR -5..10 dB
    g = np.random.default_rng(seed)
    return (torch.tensor(g.uniform(12, 26, n), dtype=torch.float32, device=dev),
            torch.tensor(g.uniform(-5, 10, n), dtype=torch.float32, device=dev))
ovr_va, snr_va = draw_levels(len(VA), 7); ovr_te, snr_te = draw_levels(len(TE), 8)
print(f"elapsed {(time.time()-T_START)/60:.1f} min")


train: 1260 mixtures from 420 rooms in 104s | 1.45 GB
val: 90 mixtures from 30 rooms in 5s | 0.10 GB
test: 180 mixtures from 60 rooms in 13s | 0.21 GB
elapsed 12.3 min


In [4]:
win = torch.hann_window(NFFT).to(dev)
MIC_SETS = {1: [0], 2: [0, 2], 4: [0, 1, 2, 3]}     # 2 mics = left-front + right-front (15 cm baseline)

def stft(x):
    sh = x.shape[:-1]
    X = torch.stft(x.reshape(-1, x.shape[-1]), NFFT, HOP, window=win, return_complex=True)
    return X.reshape(*sh, *X.shape[-2:])
def istft(X, length=T):
    sh = X.shape[:-2]
    return torch.istft(X.reshape(-1, *X.shape[-2:]), NFFT, HOP, window=win, length=length).reshape(*sh, length)

def make_mix(comp, ovr_db, snr_db):
    """comp (B,3,4,T) = [partners, wearer, noise]. Wearer = ovr_db above partners, noise = snr_db below partners."""
    g = torch.stack([torch.ones_like(ovr_db), 10 ** (ovr_db / 20), 10 ** (-snr_db / 20)], 1)[:, :, None, None]
    cs = comp * g
    sc = 0.1 / (cs.sum(1)[:, 0].pow(2).mean(-1).sqrt() + 1e-8)          # mixture RMS = 0.1 at the reference mic
    cs = cs * sc[:, None, None, None]
    return cs.sum(1), cs

def feats(X):
    """X (B,M,F,N) complex -> (B,N,K*F): log-magnitude of ref mic + [cos IPD, sin IPD, ILD] per extra mic."""
    ref = X[:, 0]; lm = torch.log(ref.abs() + 1e-5); fs = [lm / 4]
    for m in range(1, X.shape[1]):
        Xm = X[:, m]; ph = torch.angle(Xm * torch.conj(ref))
        fs += [torch.cos(ph), torch.sin(ph), (torch.log(Xm.abs() + 1e-5) - lm) / 2]
    f = torch.stack(fs, 1)
    return f.permute(0, 3, 1, 2).reshape(f.shape[0], f.shape[3], -1)

class Net(nn.Module):
    def __init__(s, M, causal, hid=HID):
        super().__init__()
        s.inp = nn.Sequential(nn.Linear((1 + 3 * (M - 1)) * NB, hid), nn.LayerNorm(hid), nn.PReLU())
        s.rnn = nn.GRU(hid, hid, num_layers=2, batch_first=True, bidirectional=not causal)
        s.out = nn.Sequential(nn.Linear(hid * (1 if causal else 2), hid), nn.PReLU(), nn.Linear(hid, NB), nn.Sigmoid())
    def forward(s, f): return s.out(s.rnn(s.inp(f))[0]).transpose(1, 2)    # mask (B,F,N)

def si_sdr(est, ref, eps=1e-8):
    est = est - est.mean(-1, keepdim=True); ref = ref - ref.mean(-1, keepdim=True)
    t = (est * ref).sum(-1, keepdim=True) / (ref.pow(2).sum(-1, keepdim=True) + eps) * ref
    return 10 * torch.log10(t.pow(2).sum(-1) / ((est - t).pow(2).sum(-1) + eps) + eps)

@torch.no_grad()
def run_system(kind, net, M, comp, ovr, snr, bs=30, keep=False):
    """kind: 'unproc' | 'oracle' | 'nn'. Returns per-mixture metrics (+ waveforms if keep)."""
    if net is not None: net.eval()
    R = {k: [] for k in ["sisdr", "sisdri", "own_sup", "noise_sup", "tgt_loss"]}; E, Rf = [], []
    pw = lambda Z: (Z.abs() ** 2).sum((-1, -2))
    db = lambda a, b: 10 * torch.log10((a + 1e-12) / (b + 1e-12))
    for i in range(0, len(comp), bs):
        mix, cs = make_mix(comp[i:i + bs].to(dev).float(), ovr[i:i + bs], snr[i:i + bs])
        X = stft(mix); Xr = X[:, 0]; Cs = stft(cs[:, :, 0])
        if kind == "unproc": Mk = torch.ones(Xr.shape, device=dev)
        elif kind == "oracle": P = Cs.abs() ** 2; Mk = (P[:, 0] / (P.sum(1) + 1e-12)).sqrt()
        else: Mk = net(feats(X[:, MIC_SETS[M]]))
        est = istft(Mk * Xr); ref = cs[:, 0, 0]
        sd = si_sdr(est, ref)
        R["sisdr"].append(sd); R["sisdri"].append(sd - si_sdr(mix[:, 0], ref))
        R["own_sup"].append(db(pw(Cs[:, 1]), pw(Mk * Cs[:, 1])))          # wearer's voice removed (dB)
        R["noise_sup"].append(db(pw(Cs[:, 2]), pw(Mk * Cs[:, 2])))        # noise removed (dB)
        R["tgt_loss"].append(db(pw(Cs[:, 0]), pw(Mk * Cs[:, 0])))         # partners' speech attenuated (dB, 0 = ideal)
        if keep: E.append(est.cpu()); Rf.append(ref.cpu())
    out = {k: torch.cat(v).cpu().numpy() for k, v in R.items()}
    if keep: out["est"] = torch.cat(E).numpy(); out["ref"] = torch.cat(Rf).numpy()
    return out


In [5]:
def val_sisdri(net, M): return float(run_system("nn", net, M, VA, ovr_va, snr_va)["sisdri"].mean())

def train(M, causal, steps=STEPS, B=BATCH):
    torch.manual_seed(0)
    net = Net(M, causal).to(dev); mics = MIC_SETS[M]
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps, eta_min=1e-4)
    curve, best, t0 = [], (-1e9, None), time.time()
    for step in range(1, steps + 1):
        net.train()
        c = TR[torch.randint(0, len(TR), (B,))].to(dev).float()
        ovr = torch.empty(B, device=dev).uniform_(5, 30); snr = torch.empty(B, device=dev).uniform_(-6, 12)
        mix, cs = make_mix(c, ovr, snr)
        X = stft(mix[:, mics]); Y = net(feats(X)) * X[:, 0]; est = istft(Y)
        ref = cs[:, 0, 0]; S = stft(ref)
        l1 = (Y.abs().clamp_min(1e-8) ** 0.3 - S.abs().clamp_min(1e-8) ** 0.3).abs().mean()
        loss = -0.05 * si_sdr(est, ref).mean() + l1
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step(); sch.step()
        if step % EVAL_EVERY == 0 or step == steps:
            v = val_sisdri(net, M); curve.append((step, v))
            if v > best[0]: best = (v, copy.deepcopy(net.state_dict()))
            print(f"   step {step:5d} | loss {loss.item():7.3f} | val SI-SDRi {v:6.2f} dB")
    net.load_state_dict(best[1]); net.eval()
    return net, curve, time.time() - t0

TRAIN_SPECS = [("m1_causal", 1, True), ("m2_causal", 2, True), ("m4_causal", 4, True), ("m1_noncausal", 1, False), ("m4_noncausal", 4, False)]
MODELS = {}
for sid, M, causal in TRAIN_SPECS:
    print(f"== training {sid} ({M} mic{'s' if M > 1 else ''}, {'causal' if causal else 'non-causal'})")
    net, curve, secs = train(M, causal)
    MODELS[sid] = dict(net=net, M=M, causal=causal, curve=curve, secs=secs, params=sum(p.numel() for p in net.parameters()))
    print(f"   done in {secs:.0f}s | params {MODELS[sid]['params']/1e6:.2f} M | elapsed {(time.time()-T_START)/60:.1f} min")


== training m1_causal (1 mic, causal)
   step   250 | loss   0.432 | val SI-SDRi  11.68 dB
   step   500 | loss   0.475 | val SI-SDRi  11.63 dB
   step   750 | loss   0.296 | val SI-SDRi  12.70 dB
   step  1000 | loss   0.369 | val SI-SDRi  12.92 dB
   step  1250 | loss   0.322 | val SI-SDRi  13.63 dB
   step  1500 | loss   0.291 | val SI-SDRi  13.95 dB
   step  1750 | loss   0.300 | val SI-SDRi  14.11 dB
   step  2000 | loss   0.330 | val SI-SDRi  14.06 dB
   step  2250 | loss   0.206 | val SI-SDRi  14.31 dB
   step  2500 | loss   0.215 | val SI-SDRi  14.27 dB
   step  2750 | loss   0.261 | val SI-SDRi  14.30 dB
   step  3000 | loss   0.263 | val SI-SDRi  14.35 dB
   done in 117s | params 0.99 M | elapsed 14.4 min
== training m2_causal (2 mics, causal)
   step   250 | loss   0.339 | val SI-SDRi  12.55 dB
   step   500 | loss   0.341 | val SI-SDRi  12.82 dB
   step   750 | loss   0.348 | val SI-SDRi  12.81 dB
   step  1000 | loss   0.279 | val SI-SDRi  13.43 dB
   step  1250 | loss   0

In [6]:
from pystoi import stoi
from pesq import pesq

def perceptual(ref, est):
    st, pq = [], []
    for r, e in zip(ref, est):
        g = 0.9 / max(np.abs(r).max(), np.abs(e).max(), 1e-9); r, e = r * g, e * g
        st.append(stoi(r, e, FS, extended=False))
        try: pq.append(pesq(FS, r, e, "wb"))
        except Exception: pq.append(np.nan)
    return np.array(st), np.array(pq)

ORDER = ["unproc", "oracle", "m1_causal", "m2_causal", "m4_causal", "m1_noncausal", "m4_noncausal"]
LABEL = dict(unproc="Unprocessed", oracle="Oracle mask (upper bound)", m1_causal="1 mic · causal", m2_causal="2 mics · causal",
             m4_causal="4 mics · causal", m1_noncausal="1 mic · non-causal", m4_noncausal="4 mics · non-causal")
def run_id(sid, comp=TE, ovr=ovr_te, snr=snr_te, keep=False):
    if sid in ("unproc", "oracle"): return run_system(sid, None, 0, comp, ovr, snr, keep=keep)
    return run_system("nn", MODELS[sid]["net"], MODELS[sid]["M"], comp, ovr, snr, keep=keep)

main = {}
for sid in ORDER:
    o = run_id(sid, keep=True); o["stoi"], o["pesq"] = perceptual(o["ref"], o["est"]); main[sid] = o
    print(f"{LABEL[sid]:28s} SI-SDRi {o['sisdri'].mean():6.2f} | STOI {o['stoi'].mean():.3f} | PESQ {np.nanmean(o['pesq']):.2f} | own-voice -{o['own_sup'].mean():5.1f} dB")

# ---- RQ3 sweeps (SI-SDR improvement), re-mixing the SAME test components at controlled levels
SW_IDS = ["oracle", "m1_causal", "m2_causal", "m4_causal"]
OVR_X, SNR_X = [0, 5, 10, 15, 20, 25, 30], [-10, -5, 0, 5, 10, 15]
def sweep(xs, which):
    ser = {sid: [] for sid in SW_IDS}
    for x in xs:
        o_, s_ = (torch.full_like(ovr_te, x), snr_te) if which == "ovr" else (ovr_te, torch.full_like(snr_te, x))
        for sid in SW_IDS: ser[sid].append(float(run_id(sid, ovr=o_, snr=s_)["sisdri"].mean()))
    return dict(x=xs, series=ser)
sw_ovr, sw_snr = sweep(OVR_X, "ovr"), sweep(SNR_X, "snr")

# ---- breakdowns by noise type and nominal RT60 (from the main test run)
env_names = sorted(meta_te.env.unique()); rt_edges = [0.2, 0.4, 0.6, 0.8]
rt_lab = [f"{a:.1f}-{b:.1f} s" for a, b in zip(rt_edges[:-1], rt_edges[1:])]
rt_bin = np.clip(np.digitize(meta_te.rt60.values, rt_edges[1:-1]), 0, 2)
def group(sid, key, vals, groups): return [float(np.nanmean(main[sid][key][vals == g])) if (vals == g).any() else None for g in groups]
by_env = {sid: dict(sisdri=group(sid, "sisdri", meta_te.env.values, env_names), stoi=group(sid, "stoi", meta_te.env.values, env_names)) for sid in SW_IDS}
by_rt = {sid: dict(sisdri=group(sid, "sisdri", rt_bin, [0, 1, 2])) for sid in SW_IDS}

# ---- inference speed of the causal 4-mic model (whole 3 s clip, batch 1, GPU/CPU)
def ms_per_audio_second(sid):
    net, M = MODELS[sid]["net"], MODELS[sid]["M"]
    mix, _ = make_mix(TE[:1].to(dev).float(), ovr_te[:1], snr_te[:1])
    with torch.no_grad():
        f = feats(stft(mix[:, MIC_SETS[M]]))
        for _ in range(3): net(f)
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20): net(f)
        if dev.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / 20 / SEG_S * 1000
speed = {sid: ms_per_audio_second(sid) for sid in ["m1_causal", "m4_causal", "m4_noncausal"]}
print("ms of compute per second of audio:", {k: round(v, 1) for k, v in speed.items()})
print(f"elapsed {(time.time()-T_START)/60:.1f} min")


Unprocessed                  SI-SDRi   0.00 | STOI 0.193 | PESQ 1.07 | own-voice -  0.0 dB
Oracle mask (upper bound)    SI-SDRi  20.50 | STOI 0.880 | PESQ 2.46 | own-voice - 23.9 dB
1 mic · causal               SI-SDRi  13.83 | STOI 0.336 | PESQ 1.07 | own-voice - 26.4 dB
2 mics · causal              SI-SDRi  15.12 | STOI 0.423 | PESQ 1.08 | own-voice - 27.4 dB
4 mics · causal              SI-SDRi  15.19 | STOI 0.430 | PESQ 1.08 | own-voice - 27.5 dB
1 mic · non-causal           SI-SDRi  14.38 | STOI 0.355 | PESQ 1.07 | own-voice - 26.6 dB
4 mics · non-causal          SI-SDRi  15.83 | STOI 0.457 | PESQ 1.09 | own-voice - 27.5 dB
ms of compute per second of audio: {'m1_causal': 1.6, 'm4_causal': 1.6, 'm4_noncausal': 3.1}
elapsed 29.1 min


In [7]:
def summ(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return dict(mean=float(a.mean()), ci=float(1.96 * a.std(ddof=1) / np.sqrt(len(a))) if len(a) > 1 else 0.0)

systems = []
for sid in ORDER:
    o, m = main[sid], MODELS.get(sid)
    systems.append(dict(id=sid, label=LABEL[sid], mics=(m["M"] if m else None), causal=(m["causal"] if m else None),
        params=(m["params"] if m else None), latency_ms=(None if not m else (32.0 if m["causal"] else SEG_S * 1000.0)),
        ms_per_s=speed.get(sid), sisdr=summ(o["sisdr"]), sisdri=summ(o["sisdri"]), stoi=summ(o["stoi"]), pesq=summ(o["pesq"]),
        own_sup=summ(o["own_sup"]), noise_sup=summ(o["noise_sup"]), tgt_loss=summ(o["tgt_loss"])))

# ---- one example: spectrograms + audio (level-matched) for the 'Listen' slide
ex_i = int(np.argsort(main["m4_causal"]["sisdri"])[len(TE) // 2])            # median-improvement test clip
ex_ids = ["unproc", "ref", "m1_causal", "m4_causal", "m4_noncausal"]
ex_lab = ["Mixture (reference mic)", "Target: partners only", "1 mic · causal", "4 mics · causal", "4 mics · non-causal"]
ex_wav = [main["unproc"]["est"][ex_i], main["unproc"]["ref"][ex_i]] + [main[s]["est"][ex_i] for s in ex_ids[2:]]
def lvl(x): return np.clip(x / (np.sqrt(np.mean(x ** 2)) + 1e-9) * 0.06, -1, 1)
def b64wav(x):
    b = io.BytesIO(); sf.write(b, x.astype(np.float32), FS, format="WAV", subtype="PCM_16"); return base64.b64encode(b.getvalue()).decode()
def spec_db(x):
    X = stft(torch.tensor(x, device=dev, dtype=torch.float32)[None])[0]
    d = 20 * torch.log10(X.abs() + 1e-6)
    return nn.functional.adaptive_avg_pool2d(d[None, None], (64, 94))[0, 0].cpu().numpy().round().astype(int).tolist()
ex_wav = [lvl(x) for x in ex_wav]
example = dict(labels=ex_lab, ids=ex_ids, audio=[b64wav(x) for x in ex_wav], spec=[spec_db(x) for x in ex_wav],
               info=dict(env=str(meta_te.env.iloc[ex_i]), rt60=float(meta_te.rt60.iloc[ex_i]), ovr=float(ovr_te[ex_i]), snr=float(snr_te[ex_i]),
                         sisdri={s: float(main[s]["sisdri"][ex_i]) for s in ["m1_causal", "m4_causal", "m4_noncausal"]}))

STEP1 = dict(utts=2703, speakers=40, hours=5.39, sr=16000, rms_dbfs=-24.6,
    demand=[dict(env="DLIVING", name="Living room", level=-56.1, corr=0.872), dict(env="OMEETING", name="Meeting room", level=-51.1, corr=0.869),
            dict(env="PCAFETER", name="Cafeteria", level=-45.1, corr=0.901), dict(env="STRAFFIC", name="Street traffic", level=-34.6, corr=0.871)],
    pilot=dict(n=30, ovr=18.8, stoi=0.291, pesq=1.112, rt60=0.46, snr=2.5, drr=-2.9,
               stoi_env=dict(DLIVING=0.287, OMEETING=0.287, PCAFETER=0.259, STRAFFIC=0.305)))

R = dict(systems=systems, curves={s: dict(steps=[c[0] for c in MODELS[s]["curve"]], val=[c[1] for c in MODELS[s]["curve"]]) for s in MODELS},
    sweeps=dict(ovr=sw_ovr, snr=sw_snr), by_env=dict(names=env_names, series=by_env), by_rt=dict(names=rt_lab, series=by_rt),
    labels={k: LABEL[k] for k in ORDER}, example=example, step1=STEP1,
    meta=dict(n_train=len(TR), n_val=len(VA), n_test=len(TE), seg_s=SEG_S, steps=STEPS, batch=BATCH, nfft=NFFT, hop=HOP, win_ms=1000 * NFFT / FS,
              spk=dict((k, len(v)) for k, v in SPK.items()), gpu=(torch.cuda.get_device_name(0) if dev.type == "cuda" else "CPU"),
              minutes=round((time.time() - T_START) / 60, 1)))
json.dump(R, open("results.json", "w"))
print("results.json saved |", f"{len(json.dumps(R))/1e6:.2f} MB")


results.json saved | 0.79 MB


In [8]:
# Presentation template (self-contained HTML/CSS/JS; results are injected below)
TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Hearing the conversation: multi-microphone enhancement on glasses</title>
<style>
:root{--blue:#2355cc;--grey:#a3a3a3;--yellow:#ebbf23;--teal:#64a8b0;--purple:#7870c4;--ink:#0b0b0c;--soft:#f3f3f3;--line:#e2e2e2;
--ease:cubic-bezier(.77,0,.18,1)}
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;overflow:hidden;background:#fff;color:var(--ink);font-family:"Inter","Helvetica Neue",Arial,system-ui,sans-serif;-webkit-font-smoothing:antialiased}
button{font:inherit;color:inherit;cursor:pointer;background:none;border:0}
#deck{position:fixed;inset:0;overflow:hidden}
#track{height:100%;transition:transform .95s var(--ease);will-change:transform}
.slide{height:100vh;width:100%;position:relative;padding:96px 8vw 76px 6vw;display:flex;flex-direction:column;justify-content:center;overflow:hidden}
.slide>.grid{flex:1;min-height:0}
.r{opacity:0;transform:translateY(26px);transition:opacity .7s ease,transform .8s var(--ease)}
.slide.on .r{opacity:1;transform:none;transition-delay:calc(var(--i,0)*90ms + 420ms)}
/* chrome */
.brand{position:fixed;left:6vw;top:26px;z-index:20;font-size:12px;letter-spacing:.14em;text-transform:uppercase;font-weight:600;display:flex;align-items:center;gap:10px}
.brand i{display:block;width:34px;height:10px;background:repeating-linear-gradient(135deg,var(--yellow) 0 4px,transparent 4px 8px);border:1.5px solid var(--ink)}
.gh{position:fixed;right:64px;top:18px;z-index:20;border:1.5px solid var(--ink);border-radius:999px;padding:9px 18px;font-size:13px;font-weight:600;background:#fff;transition:.25s;display:flex;gap:8px;align-items:center;text-decoration:none;color:var(--ink)}
.gh:hover{background:var(--blue);border-color:var(--blue);color:#fff;transform:translateY(-2px)}
.foot{position:fixed;left:6vw;bottom:24px;z-index:20;font-size:12px;color:var(--grey);display:flex;gap:18px;align-items:center}
.foot b{color:var(--ink);font-weight:600}
#count{font-variant-numeric:tabular-nums;letter-spacing:.08em}
#dots{position:fixed;right:22px;top:50%;transform:translateY(-50%);z-index:30;display:flex;flex-direction:column;gap:15px;align-items:center}
#dots:before{content:"";position:absolute;top:6px;bottom:6px;width:2px;background:var(--line);z-index:-1}
#dots .fill{position:absolute;top:6px;width:2px;background:var(--blue);z-index:-1;transition:height .9s var(--ease)}
.dot{position:relative;width:12px;height:12px;border-radius:50%;background:#fff;border:2px solid var(--grey);transition:.35s}
.dot:hover{border-color:var(--ink);transform:scale(1.25)}
.dot.on{background:var(--blue);border-color:var(--blue);box-shadow:0 0 0 5px rgba(35,85,204,.16);transform:scale(1.2)}
.dot span{position:absolute;right:26px;top:50%;transform:translateY(-50%) translateX(8px);white-space:nowrap;background:var(--ink);color:#fff;font-size:11px;padding:5px 10px;border-radius:6px;opacity:0;pointer-events:none;transition:.25s}
.dot:hover span{opacity:1;transform:translateY(-50%)}
/* type */
.kick{display:inline-flex;align-items:center;gap:10px;font-size:12px;letter-spacing:.16em;text-transform:uppercase;font-weight:700;color:var(--blue)}
.kick:before{content:"";width:28px;height:8px;background:repeating-linear-gradient(135deg,var(--blue) 0 3px,transparent 3px 6px)}
h1{font-size:clamp(38px,5.3vw,78px);line-height:1.02;letter-spacing:-.035em;font-weight:800}
h2{font-size:clamp(28px,3.3vw,48px);line-height:1.06;letter-spacing:-.03em;font-weight:800;margin:10px 0 12px}
h1 em,h2 em{font-style:normal;color:var(--blue)}
.mark{background:linear-gradient(transparent 62%,var(--yellow) 62% 92%,transparent 92%);padding:0 .08em}
.lead{font-size:clamp(15px,1.25vw,19px);line-height:1.55;color:#3a3a3a;max-width:60ch}
.small{font-size:13px;line-height:1.5;color:#6b6b6b}
.ghost{-webkit-text-stroke:1.5px var(--blue);color:transparent;background:repeating-linear-gradient(135deg,var(--blue) 0 2px,transparent 2px 6px);-webkit-background-clip:text;background-clip:text;font-weight:800}
.num{font-variant-numeric:tabular-nums}
/* components */
.card{border:1.5px solid var(--ink);border-radius:16px;background:#fff;padding:18px 20px;box-shadow:5px 5px 0 var(--ink);transition:.3s var(--ease)}
.card:hover{transform:translate(-3px,-3px);box-shadow:8px 8px 0 var(--blue)}
.chip{border:1.5px solid var(--ink);border-radius:999px;padding:7px 14px;font-size:13px;font-weight:600;background:#fff;transition:.25s}
.chip:hover{background:var(--soft)} .chip.on{background:var(--ink);color:#fff}
.chips{display:flex;gap:8px;flex-wrap:wrap}
.btn{display:inline-flex;align-items:center;gap:10px;background:var(--blue);color:#fff;border-radius:999px;padding:13px 24px;font-weight:700;font-size:15px;border:1.5px solid var(--blue);transition:.25s;text-decoration:none}
.btn:hover{background:#fff;color:var(--blue);transform:translateY(-2px)}
.btn.alt{background:#fff;color:var(--ink);border-color:var(--ink)} .btn.alt:hover{background:var(--ink);color:#fff}
.two{display:grid;grid-template-columns:1fr 1.15fr;gap:5vw;align-items:center;height:100%}
.two.rev{grid-template-columns:1.1fr 1fr}
.stack>*+*{margin-top:16px}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:12.5px;font-weight:600}
.legend i{display:inline-block;width:11px;height:11px;border-radius:3px;margin-right:6px;vertical-align:-1px}
.chart{position:relative;width:100%;height:100%;min-height:260px}
.chart svg{width:100%;height:100%;overflow:visible;display:block}
.plotbox{border:1.5px solid var(--ink);border-radius:18px;padding:16px 18px 10px;background:#fff;height:100%;display:flex;flex-direction:column;gap:10px;position:relative}
.plotbox .chart{flex:1}
.plotbox:before{content:"";position:absolute;right:-1.5px;top:22px;width:8px;height:56px;background:repeating-linear-gradient(135deg,var(--yellow) 0 3px,var(--teal) 3px 6px,var(--purple) 6px 9px);border:1.5px solid var(--ink);border-left:0;border-radius:0 6px 6px 0}
#tip{position:fixed;pointer-events:none;z-index:50;background:var(--ink);color:#fff;padding:8px 12px;border-radius:9px;font-size:12px;line-height:1.45;opacity:0;transform:translateY(6px);transition:opacity .15s,transform .15s;max-width:260px}
#tip.show{opacity:1;transform:none} #tip b{color:var(--yellow)}
.ax text{font-size:11px;fill:#6b6b6b;font-variant-numeric:tabular-nums}.ax line,.ax path{stroke:var(--line)}
.lbl{font-size:12px;font-weight:600;fill:var(--ink)}
.bignum{font-size:clamp(34px,3.6vw,56px);font-weight:800;letter-spacing:-.04em;line-height:1}
.stat{border-top:2px solid var(--ink);padding-top:10px}
.stat .small{margin-top:4px}
.rq{display:flex;gap:16px;align-items:flex-start}
.rq .tag{flex:none;width:54px;height:54px;border-radius:14px;border:1.5px solid var(--ink);display:grid;place-items:center;font-weight:800;font-size:15px}
.flow{display:flex;align-items:stretch;gap:10px;flex-wrap:wrap}
.node{flex:1;min-width:118px;border:1.5px solid var(--ink);border-radius:14px;padding:14px;background:#fff;position:relative;transition:.3s var(--ease);cursor:default}
.node:hover{transform:translateY(-6px);background:var(--ink);color:#fff}
.node:hover .small{color:#ddd}
.node b{display:block;font-size:14px;margin-bottom:4px}
.node:not(:last-child):after{content:"→";position:absolute;right:-17px;top:50%;transform:translateY(-50%);font-weight:800;color:var(--blue);z-index:2}
.metric-row{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}
audio{display:none}
.spec{border:1.5px solid var(--ink);border-radius:14px;overflow:hidden;background:#fff;transition:.3s var(--ease);cursor:pointer;position:relative}
.spec:hover{transform:translateY(-5px);box-shadow:0 8px 0 var(--blue)}
.spec canvas{width:100%;height:150px;display:block}
.spec .cap{display:flex;justify-content:space-between;align-items:center;padding:9px 12px;border-top:1.5px solid var(--ink);font-size:12.5px;font-weight:700}
.play{width:26px;height:26px;border-radius:50%;background:var(--blue);display:grid;place-items:center;color:#fff;font-size:10px}
.spec.playing .play{background:var(--yellow);color:var(--ink)}
.pill{display:inline-block;border-radius:999px;padding:3px 10px;font-size:11.5px;font-weight:700;border:1.5px solid var(--ink)}
@media (max-width:900px){.two,.two.rev{grid-template-columns:1fr;gap:20px;height:auto}.slide{overflow-y:auto;padding-top:84px}.metric-row{grid-template-columns:1fr}.gh{right:56px}.node:not(:last-child):after{display:none}}
@media (prefers-reduced-motion:reduce){#track{transition:none}.r{transition:none}}

</style></head><body>
<div class="brand"><i></i>Speech enhancement · side project</div>
<a class="gh" href="https://github.com/alaaguedda" target="_blank" rel="noopener"><svg width="16" height="16" viewBox="0 0 16 16" fill="currentColor"><path d="M8 0C3.58 0 0 3.58 0 8c0 3.54 2.29 6.53 5.47 7.59.4.07.55-.17.55-.38v-1.33c-2.23.48-2.7-1.07-2.7-1.07-.36-.92-.89-1.17-.89-1.17-.73-.5.05-.49.05-.49.8.06 1.23.83 1.23.83.71 1.22 1.87.87 2.33.66.07-.52.28-.87.5-1.07-1.78-.2-3.64-.89-3.64-3.95 0-.87.31-1.59.82-2.15-.08-.2-.36-1.02.08-2.12 0 0 .67-.21 2.2.82a7.6 7.6 0 014 0c1.53-1.03 2.2-.82 2.2-.82.44 1.1.16 1.92.08 2.12.51.56.82 1.28.82 2.15 0 3.07-1.87 3.75-3.65 3.95.29.25.54.73.54 1.48v2.2c0 .21.15.46.55.38A8 8 0 0016 8c0-4.42-3.58-8-8-8z"/></svg>Source code ↗</a>
<div class="foot"><span><b>Alaa Guedda</b></span><span id="count">01 / 10</span><span>↓ scroll · ← → keys</span></div>
<nav id="dots" aria-label="Pages"><div class="fill" id="fill"></div></nav>
<div id="tip"></div>
<main id="deck"><div id="track">

<!-- 1 TITLE -->
<section class="slide" data-title="Intro"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0">Multi-microphone · assistive listening</div>
  <h1 class="r" style="--i:1">Can a pair of <em>smart glasses</em> help you <span class="mark">hear the conversation?</span></h1>
  <p class="lead r" style="--i:2">A small, fully reproducible study on open data: four microphones on a glasses frame, a wearer who is much louder than everyone else, two conversation partners and real background noise.</p>
  <div class="r" style="--i:3;display:flex;gap:12px;flex-wrap:wrap;align-items:center"><button class="btn" data-go="1">Start the story →</button><a class="btn alt" href="https://github.com/alaaguedda" target="_blank" rel="noopener">Source code ↗</a></div>
  <p class="small r" style="--i:4">By <b style="color:var(--ink)">Alaa Guedda</b> · inspired by the ECHI-2 conversation-enhancement task</p>
 </div>
 <div class="r" style="--i:2;height:100%;min-height:340px;position:relative"><canvas id="hero" style="width:100%;height:100%"></canvas></div>
</div></section>

<!-- 2 QUESTION -->
<section class="slide" data-title="The question"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0">The problem</div>
  <h2 class="r" style="--i:1">People with hearing loss lose the <em>cocktail party</em>.</h2>
  <p class="lead r" style="--i:2">Assistive listening should lift everyone in the conversation, remove the noise, the competing talkers and the wearer's own voice, and do it fast enough to keep lips and sound in sync.</p>
  <p class="lead r" style="--i:3">The real challenge data comes from smart-glasses recordings. Here I build an open stand-in with the same ingredients and ask three focused questions.</p>
 </div>
 <div class="stack">
  <div class="card rq r" style="--i:2"><div class="tag" style="background:var(--blue);color:#fff">RQ1</div><div><b>Do more microphones help?</b><p class="small">Same network, 1 vs 2 vs 4 microphones. Does spatial information change what is possible?</p></div></div>
  <div class="card rq r" style="--i:3"><div class="tag" style="background:var(--purple);color:#fff">RQ2</div><div><b>What does low latency cost?</b><p class="small">Causal processing (32 ms) against a model that sees the whole segment.</p></div></div>
  <div class="card rq r" style="--i:4"><div class="tag" style="background:var(--yellow)">RQ3</div><div><b>Where does it break?</b><p class="small">Wearer loudness, noise level, reverberation and noise type.</p></div></div>
 </div>
</div></section>

<!-- 3 DATA -->
<section class="slide" data-title="Data"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0">Data</div>
  <h2 class="r" style="--i:1">Open data, <em>simulated</em> glasses.</h2>
  <div class="metric-row r" style="--i:2">
   <div class="stat"><div class="bignum" data-count="utts">0</div><div class="small">speech utterances</div></div>
   <div class="stat"><div class="bignum" data-count="speakers">0</div><div class="small">speakers (split by speaker)</div></div>
   <div class="stat"><div class="bignum" data-count="hours" data-dec="2">0</div><div class="small">hours of clean speech</div></div>
  </div>
  <div class="flow r" style="--i:3">
   <div class="node"><b>LibriSpeech</b><span class="small">wearer + 2 partners</span></div>
   <div class="node"><b>Room acoustics</b><span class="small">random rooms, 0.2 to 0.8 s, random head turn</span></div>
   <div class="node"><b>4-mic glasses</b><span class="small">15 cm wide, mouth 14 cm away</span></div>
   <div class="node"><b>DEMAND noise</b><span class="small">real 16-channel recordings</span></div>
  </div>
  <p class="small r" style="--i:4">Each mixture lasts 3 s. Wearer level, noise level and room change on every draw; speakers and noise segments never overlap between train and test.</p>
 </div>
 <div class="plotbox r" style="--i:3">
  <div><b>Four real noise scenes</b> <span class="small">· level in dBFS, hover for spatial coherence</span></div>
  <div class="chart" id="c-noise" style="min-height:150px"></div>
  <div class="metric-row" style="border-top:1.5px solid var(--ink);padding-top:12px">
   <div><div class="bignum" id="p-ovr">–</div><div class="small">dB: wearer louder than partners</div></div>
   <div><div class="bignum" id="p-stoi">–</div><div class="small">STOI before processing</div></div>
   <div><div class="bignum" id="p-pesq">–</div><div class="small">PESQ before processing</div></div>
  </div>
  <p class="small">Pilot on 30 mixtures: the unprocessed signal is almost unintelligible. The wearer's voice is the real enemy.</p>
 </div>
</div></section>

<!-- 4 METHOD -->
<section class="slide" data-title="Method"><div class="two rev">
 <div class="stack">
  <div class="kick r" style="--i:0">Method</div>
  <h2 class="r" style="--i:1">One small network, <em>spatial cues</em> as input.</h2>
  <div class="flow r" style="--i:2">
   <div class="node"><b>STFT</b><span class="small" id="m-stft">32 ms window · 16 ms hop</span></div>
   <div class="node"><b>Features</b><span class="small">level + phase differences between mics</span></div>
   <div class="node"><b>2-layer GRU</b><span class="small">causal or bidirectional</span></div>
   <div class="node"><b>Mask</b><span class="small">one gain per time-frequency bin</span></div>
   <div class="node"><b>iSTFT</b><span class="small">enhanced reference mic</span></div>
  </div>
  <div class="r" style="--i:3"><div class="small" style="margin-bottom:8px;font-weight:700;color:var(--ink)">Pick an array</div>
   <div class="chips" id="m-chips"><button class="chip on" data-m="1">1 mic</button><button class="chip" data-m="2">2 mics</button><button class="chip" data-m="4">4 mics</button></div></div>
  <div class="metric-row r" style="--i:4">
   <div class="stat"><div class="bignum" id="m-maps">1</div><div class="small">feature maps per bin</div></div>
   <div class="stat"><div class="bignum" id="m-dim">257</div><div class="small">inputs per frame</div></div>
   <div class="stat"><div class="bignum" id="m-par">–</div><div class="small">M parameters (causal)</div></div>
  </div>
 </div>
 <div class="stack">
  <div class="card r" style="--i:3"><b>Target</b><p class="small">The partners' reverberant speech at the reference microphone. The wearer and the noise must disappear.</p></div>
  <div class="card r" style="--i:4"><b>Training</b><p class="small" id="m-train">–</p></div>
  <div class="card r" style="--i:5"><b>Metrics</b><p class="small">SI-SDR improvement, STOI, PESQ, plus how many dB of wearer voice and noise are removed and how much partner speech is lost.</p></div>
 </div>
</div></section>

<!-- 5 RQ1 -->
<section class="slide" data-title="RQ1 · Microphones"><div class="two rev">
 <div class="stack">
  <div class="kick r" style="--i:0"><span class="ghost" style="font-size:20px">RQ1</span> Do more microphones help?</div>
  <h2 class="r" style="--i:1" id="rq1-h">–</h2>
  <p class="lead r" style="--i:2" id="rq1-t">–</p>
  <div class="chips r" style="--i:3" id="rq1-chips"></div>
  <div class="legend r" style="--i:4" id="rq1-leg"></div>
 </div>
 <div style="display:grid;grid-template-rows:1.3fr 1fr;gap:16px;height:100%;min-height:0">
  <div class="plotbox r" style="--i:2"><div class="chart" id="c-rq1"></div></div>
  <div class="plotbox r" style="--i:3"><div class="small"><b style="color:var(--ink)">Learning curves</b> · validation SI-SDR improvement during training</div><div class="chart" id="c-curve"></div></div>
 </div>
</div></section>

<!-- 6 RQ2 -->
<section class="slide" data-title="RQ2 · Latency"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0"><span class="ghost" style="font-size:20px">RQ2</span> What does low latency cost?</div>
  <h2 class="r" style="--i:1" id="rq2-h">–</h2>
  <p class="lead r" style="--i:2" id="rq2-t">–</p>
  <div class="metric-row r" style="--i:3">
   <div class="stat"><div class="bignum" style="color:var(--blue)">32<span style="font-size:.45em"> ms</span></div><div class="small">causal: one STFT window, no look-ahead</div></div>
   <div class="stat"><div class="bignum" style="color:var(--purple)" id="rq2-nc">3<span style="font-size:.45em"> s</span></div><div class="small">non-causal: needs the whole segment</div></div>
   <div class="stat"><div class="bignum" id="rq2-sp">–</div><div class="small">ms compute per second of audio (4 mics, causal)</div></div>
  </div>
 </div>
 <div class="plotbox r" style="--i:3"><div class="chips" id="rq2-chips"></div><div class="chart" id="c-rq2"></div>
  <div class="legend"><span><i style="background:var(--blue)"></i>Causal</span><span><i style="background:var(--purple)"></i>Non-causal</span></div></div>
</div></section>

<!-- 7 RQ3 -->
<section class="slide" data-title="RQ3 · Robustness"><div class="two rev">
 <div class="stack">
  <div class="kick r" style="--i:0"><span class="ghost" style="font-size:20px">RQ3</span> Where does it break?</div>
  <h2 class="r" style="--i:1" id="rq3-h">–</h2>
  <p class="lead r" style="--i:2" id="rq3-t">–</p>
  <div class="chips r" style="--i:3" id="rq3-chips"></div>
  <div class="legend r" style="--i:4" id="rq3-leg"></div>
 </div>
 <div class="plotbox r" style="--i:3"><div class="small" id="rq3-cap"></div><div class="chart" id="c-rq3"></div></div>
</div></section>

<!-- 8 ANATOMY -->
<section class="slide" data-title="Anatomy"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0">Anatomy of the enhancement</div>
  <h2 class="r" style="--i:1" id="an-h">–</h2>
  <p class="lead r" style="--i:2" id="an-t">–</p>
  <p class="small r" style="--i:3">Measured by applying each model's mask separately to the wearer's voice, the noise and the partners' speech.</p>
 </div>
 <div class="plotbox r" style="--i:3"><div class="chart" id="c-an"></div><div class="legend" id="an-leg"></div></div>
</div></section>

<!-- 9 LISTEN -->
<section class="slide" data-title="Listen"><div style="display:flex;flex-direction:column;gap:18px;height:100%;justify-content:center">
  <div class="stack"><div class="kick r" style="--i:0">Listen</div>
  <h2 class="r" style="--i:1">One test clip, <em>five versions</em>.</h2>
  <p class="small r" style="--i:2" id="ex-info">–</p></div>
  <div id="specs" class="r" style="--i:3;display:grid;grid-template-columns:repeat(5,1fr);gap:14px"></div>
  <p class="small r" style="--i:4">Click a card to play it. Every clip is level-matched. Spectrograms show 0 to 8 kHz, low frequencies at the bottom.</p>
</div></section>

<!-- 10 TAKEAWAYS -->
<section class="slide" data-title="Takeaways"><div class="two">
 <div class="stack">
  <div class="kick r" style="--i:0">Takeaways</div>
  <h2 class="r" style="--i:1">What this <em>shows</em>.</h2>
  <div class="stack" id="take"></div>
  <div class="r" style="--i:5"><div class="small" style="font-weight:700;color:var(--ink);margin-bottom:6px">Honest limits</div>
   <p class="small">Simulated array and rooms. The noise comes from a different array, so it is spatially coherent (inter-channel correlation about 0.87 to 0.90). One training run per model, mask applied to one reference microphone, no binaural rendering.</p></div>
 </div>
 <div class="stack">
  <div class="card r" style="--i:3"><b>Next, with the real glasses data</b>
   <p class="small" style="margin-top:6px">Binaural rendering, spatially selective deep filters, stronger backbones such as TF-GridNet, hybrid model-based plus neural separation, and the multimodal track with video and motion sensors.</p></div>
  <div class="card r" style="--i:4;border-color:var(--blue);box-shadow:5px 5px 0 var(--blue)"><div style="display:flex;justify-content:space-between;gap:12px;align-items:center;flex-wrap:wrap">
   <div><div class="small">Author</div><div style="font-weight:800;font-size:22px;letter-spacing:-.02em">Alaa Guedda</div></div>
   <a class="btn" href="https://github.com/alaaguedda" target="_blank" rel="noopener">Source code ↗</a></div></div>
 </div>
</div></section>

</div></main>

<script>
const R = /*__RESULTS__*/null;
const $=(s,r=document)=>r.querySelector(s), $$=(s,r=document)=>[...r.querySelectorAll(s)];
const NS="http://www.w3.org/2000/svg";
const COL={blue:"#2355cc",grey:"#a3a3a3",yellow:"#ebbf23",teal:"#64a8b0",purple:"#7870c4",ink:"#0b0b0c"};
const SC={unproc:COL.ink,oracle:COL.yellow,m1_causal:COL.grey,m2_causal:COL.teal,m4_causal:COL.blue,m1_noncausal:COL.purple,m4_noncausal:COL.purple};
const SYS=Object.fromEntries(R.systems.map(s=>[s.id,s]));
const f1=v=>(v>=0?"":"−")+Math.abs(v).toFixed(1), f2=v=>v.toFixed(2), f3=v=>v.toFixed(3);
const sgn=v=>(v>=0?"+":"−")+Math.abs(v).toFixed(1);
function el(tag,at={},parent){const e=document.createElementNS(NS,tag);for(const k in at)e.setAttribute(k,at[k]);if(parent)parent.appendChild(e);return e}
function mount(id){const box=$("#"+id);box.innerHTML="";const w=box.clientWidth||600,h=box.clientHeight||300;
  const svg=el("svg",{viewBox:`0 0 ${w} ${h}`},box);return{box,svg,w,h}}
const tip=$("#tip");
function showTip(e,html){tip.innerHTML=html;tip.classList.add("show");
  tip.style.left=Math.min(e.clientX+14,innerWidth-tip.offsetWidth-10)+"px";tip.style.top=Math.min(e.clientY+14,innerHeight-tip.offsetHeight-10)+"px"}
const hideTip=()=>tip.classList.remove("show");
function niceTicks(lo,hi,n=5){const span=hi-lo||1,step0=span/n,mag=Math.pow(10,Math.floor(Math.log10(step0))),
  step=[1,2,2.5,5,10].map(m=>m*mag).find(s=>s>=step0)||step0;const a=Math.floor(lo/step)*step,t=[];for(let v=a;v<=hi+step*.001;v+=step)t.push(+v.toFixed(6));return t}
function chips(id,items,onPick,start=0){const box=$("#"+id);box.innerHTML="";items.forEach((it,i)=>{const b=document.createElement("button");
  b.className="chip"+(i===start?" on":"");b.textContent=it.label;b.onclick=()=>{$$(".chip",box).forEach(c=>c.classList.remove("on"));b.classList.add("on");onPick(it)};box.appendChild(b)});onPick(items[start])}
function legend(id,ids,labels){$("#"+id).innerHTML=ids.map(k=>`<span><i style="background:${SC[k]};${k==="unproc"?"":""}"></i>${labels?labels[k]:R.labels[k]}</span>`).join("")}
function countUp(node,to,dec=0,ms=1300){const t0=performance.now();(function s(n){const p=Math.min(1,(n-t0)/ms),e=1-Math.pow(1-p,3);
  node.textContent=(to*e).toLocaleString("en-US",{minimumFractionDigits:dec,maximumFractionDigits:dec});if(p<1)requestAnimationFrame(s)})(t0)}

/* ---------- page navigation ---------- */
const slides=$$(".slide"),track=$("#track"),N=slides.length;let cur=-1,lock=false,lockT=0,wt;
slides.forEach((s,i)=>{const d=document.createElement("button");d.className="dot";d.setAttribute("aria-label",s.dataset.title);
  d.innerHTML=`<span>${String(i+1).padStart(2,"0")} · ${s.dataset.title}</span>`;d.onclick=()=>go(i);$("#dots").appendChild(d)});
const hooks={};
function go(i){i=Math.max(0,Math.min(N-1,i));if(i===cur)return;cur=i;track.style.transform=`translateY(${-100*i}vh)`;
  slides.forEach((s,k)=>s.classList.toggle("on",k===i));$$(".dot").forEach((d,k)=>d.classList.toggle("on",k===i));
  $("#count").textContent=String(i+1).padStart(2,"0")+" / "+String(N).padStart(2,"0");
  $("#fill").style.height=Math.max(0,($("#dots").clientHeight-12)*i/(N-1))+"px";hideTip();
  if(hooks[i])setTimeout(()=>hooks[i](),380)}
addEventListener("wheel",e=>{e.preventDefault();if(Math.abs(e.deltaY)<4)return;const now=Date.now();
  if(!lock||now-lockT>1300){lock=true;lockT=now;go(cur+Math.sign(e.deltaY))}clearTimeout(wt);wt=setTimeout(()=>lock=false,650)},{passive:false});
addEventListener("keydown",e=>{const k=e.key;if(["ArrowDown","ArrowRight","PageDown"," "].includes(k)){e.preventDefault();go(cur+1)}
  else if(["ArrowUp","ArrowLeft","PageUp"].includes(k)){e.preventDefault();go(cur-1)}else if(k==="Home")go(0);else if(k==="End")go(N-1)});
let ty=0;addEventListener("touchstart",e=>ty=e.touches[0].clientY,{passive:true});
addEventListener("touchend",e=>{const d=ty-e.changedTouches[0].clientY;if(Math.abs(d)>50)go(cur+Math.sign(d))},{passive:true});
document.addEventListener("click",e=>{const b=e.target.closest("[data-go]");if(b)go(+b.dataset.go)});


/* ---------- hero: moving array drawing ---------- */

function hero(){
  const c=$("#hero"),x=c.getContext("2d");let W,H;
  const size=()=>{const d=devicePixelRatio||1;W=c.clientWidth;H=c.clientHeight;c.width=W*d;c.height=H*d;x.setTransform(d,0,0,d,0,0)};size();addEventListener("resize",size);
  const P=Array.from({length:54},(_,i)=>({a:Math.random()*6.283,r:.86+Math.random()*.34,s:(.05+Math.random()*.16)*(i%2?1:-1),z:1.5+Math.random()*3.5}));
  const src=[{a:-.75,d:.78,c:COL.teal,n:"partner A"},{a:.62,d:.66,c:COL.purple,n:"partner B"}];
  const t0=performance.now();
  function draw(now){requestAnimationFrame(draw);if(cur>1)return;if(c.clientWidth!==W||c.clientHeight!==H)size();if(W<80||H<80)return;const t=Math.max(0,(now-t0)/1000);x.clearRect(0,0,W,H);
    const U=Math.min(W,H*1.05)/2.15,cx=W/2,cy=H*.46,hr=.2*U,pol=(a,d)=>[cx+Math.sin(a)*d*U,cy-Math.cos(a)*d*U];
    x.lineWidth=1;x.strokeStyle="#ececec";[.45,.8,1.1].forEach(r=>{x.beginPath();x.arc(cx,cy,r*U,0,6.283);x.stroke()});
    // noise particles
    P.forEach(p=>{const a=p.a+t*p.s,[px,py]=pol(a,p.r+Math.sin(t*.8+p.z)*.04);x.fillStyle="rgba(163,163,163,"+(.35+.3*Math.sin(t+p.z))+")";x.beginPath();x.arc(px,py,p.z,0,6.283);x.fill()});
    // listening lobes toward partners
    src.forEach((s,i)=>{const [sx,sy]=pol(s.a,s.d),g=x.createLinearGradient(cx,cy,sx,sy);g.addColorStop(0,"rgba(35,85,204,.0)");g.addColorStop(1,"rgba(35,85,204,"+(.16+.07*Math.sin(t*2+i))+")");
      x.fillStyle=g;x.beginPath();x.moveTo(cx,cy);x.arc(cx,cy,s.d*U*1.05,-Math.PI/2+s.a-.26,-Math.PI/2+s.a+.26);x.closePath();x.fill()});
    // rings from partners and wearer
    const ring=(sx,sy,col,R0,ph,w)=>{for(let j=0;j<3;j++){const q=(t*.42+ph+j/3)%1;x.strokeStyle=col;x.globalAlpha=(1-q)*.75;x.lineWidth=w;x.beginPath();x.arc(sx,sy,q*R0,0,6.283);x.stroke()}x.globalAlpha=1};
    src.forEach((s,i)=>{const [sx,sy]=pol(s.a,s.d);ring(sx,sy,s.c,.55*U,i*.37,2)});
    const [mx,my]=pol(0,.24);ring(mx,my,COL.yellow,.9*U,.1,3.2);
    // wearer-suppression null
    x.setLineDash([4,5]);x.strokeStyle=COL.yellow;x.lineWidth=1.8;x.beginPath();x.arc(mx,my,Math.max(1,.1*U+Math.sin(t*3)*2),0,6.283);x.stroke();x.setLineDash([]);
    // head + glasses + mics
    x.fillStyle="#fff";x.strokeStyle=COL.ink;x.lineWidth=2.4;x.beginPath();x.arc(cx,cy,hr,0,6.283);x.fill();x.stroke();
    x.beginPath();x.moveTo(cx-hr*.16,cy-hr+1);x.lineTo(cx,cy-hr*1.32);x.lineTo(cx+hr*.16,cy-hr+1);x.stroke();
    const fy=cy-hr*.42;x.lineWidth=3;x.beginPath();x.moveTo(cx-hr*1.02,fy+hr*.5);x.lineTo(cx-hr*.9,fy);x.lineTo(cx+hr*.9,fy);x.lineTo(cx+hr*1.02,fy+hr*.5);x.stroke();
    [[-.95,0],[-.95,.5],[.95,0],[.95,.5]].forEach(([dx,dy],k)=>{const px=cx+dx*hr,py=fy+dy*hr,s=6+Math.sin(t*4+k)*1.2;x.fillStyle=COL.blue;x.fillRect(px-s/2,py-s/2,s,s)});
    // sources + labels
    x.font="600 11px Inter,system-ui,sans-serif";x.textAlign="center";
    src.forEach(s=>{const [sx,sy]=pol(s.a,s.d);x.fillStyle=s.c;x.strokeStyle=COL.ink;x.lineWidth=2;x.beginPath();x.arc(sx,sy,10,0,6.283);x.fill();x.stroke();x.fillStyle=COL.ink;x.fillText(s.n.toUpperCase(),sx,sy-18)});
    x.fillStyle=COL.ink;x.fillText("WEARER",mx,my-.24*U-4);x.fillStyle="#8a8a8a";x.fillText("NOISE",...pol(-1.75,1.04));
    // waveform strip: noisy <-> enhanced
    const wy=H*.925,wa=Math.min(26,H*.05),k=(Math.sin(t*.7)+1)/2,x0=W*.1,x1=W*.9;
    x.lineWidth=2.4;x.strokeStyle=COL.blue;x.beginPath();
    for(let i=0;i<=160;i++){const u=i/160,px=x0+u*(x1-x0),clean=Math.sin(u*38+t*3)*Math.sin(u*5.5-t*.5)*.7+Math.sin(u*71+t)*.2,
      nz=(Math.sin(u*190+t*7)+Math.sin(u*263-t*5)*.8+Math.sin(u*97+t*9)*.6)*.4*(1-k)*1.4;
      const py=wy-(clean+nz)*wa;i?x.lineTo(px,py):x.moveTo(px,py)}x.stroke();
    x.fillStyle="#8a8a8a";x.textAlign="left";x.fillText("MIXTURE",x0,wy+wa+16);x.textAlign="right";x.fillText("ENHANCED",x1,wy+wa+16);
    x.fillStyle=COL.ink;x.textAlign="center";x.fillText(k<.5?"noisy":"clean",W/2,wy+wa+16)}
  requestAnimationFrame(draw)}

/* ---------- chart primitives (hand-rolled SVG) ---------- */
function barsH(id,rows,o={}){
  const {svg,w,h}=mount(id),L=Math.min(170,w*.34),Rm=64,T=8,B=26;
  const lo=Math.min(o.min??0,...rows.map(r=>r.v-(r.ci||0))),hi=Math.max(o.max??0,...rows.map(r=>r.v+(r.ci||0)))*1.04;
  const x=v=>L+(v-lo)/(hi-lo||1)*(w-L-Rm),base=x(Math.max(lo,Math.min(0,hi))),rh=(h-T-B)/rows.length,bh=Math.min(30,rh*.58);
  const ax=el("g",{class:"ax"},svg);niceTicks(lo,hi,5).forEach(t=>{if(t<lo-1e-9||t>hi)return;el("line",{x1:x(t),x2:x(t),y1:T,y2:h-B},ax);el("text",{x:x(t),y:h-8,"text-anchor":"middle"},ax).textContent=(o.tick||(v=>v))(t)});
  if(o.xlabel)el("text",{x:w-2,y:h-8,"text-anchor":"end",style:"font-size:11px;fill:#a3a3a3"},svg).textContent=o.xlabel;
  rows.forEach((r,i)=>{const y=T+i*rh+(rh-bh)/2,g=el("g",{style:"cursor:default"},svg),x1=x(r.v),bx=Math.min(base,x1),bw=Math.abs(x1-base);
    el("text",{x:L-12,y:y+bh/2+4,"text-anchor":"end",class:"lbl"},g).textContent=r.label;
    el("rect",{x:L,y:y-4,width:w-L-Rm,height:bh+8,fill:"transparent"},g);
    const b=el("rect",{x:base,y,width:0,height:bh,rx:6,fill:r.color,stroke:COL.ink,"stroke-width":1.5,style:`transition:all .9s cubic-bezier(.2,.8,.2,1) ${i*70}ms`},g);
    requestAnimationFrame(()=>requestAnimationFrame(()=>{b.setAttribute("x",bx);b.setAttribute("width",Math.max(bw,2))}));
    if(r.ci){const a=x(r.v-r.ci),c=x(r.v+r.ci),m=y+bh/2;el("path",{d:`M${a} ${m}H${c}M${a} ${m-5}V${m+5}M${c} ${m-5}V${m+5}`,stroke:COL.ink,"stroke-width":1.6,fill:"none"},g)}
    el("text",{x:(r.v>=0?Math.max(x1,x(r.v+(r.ci||0))):x1)+8,y:y+bh/2+4,class:"lbl num"},g).textContent=r.text??(o.fmt||f2)(r.v);
    g.onmousemove=e=>{b.setAttribute("opacity",.85);showTip(e,r.tip||`<b>${r.label}</b><br>${(o.fmt||f2)(r.v)}${r.ci?` ± ${(r.ci).toFixed(2)} (95% CI)`:""}`)};g.onmouseleave=()=>{b.setAttribute("opacity",1);hideTip()}})}

function lineChart(id,o){
  const {svg,w,h}=mount(id),L=46,Rm=22,T=14,B=o.xlabel?44:30,n=o.xs.length;
  const all=o.series.flatMap(s=>s.vals).filter(v=>v!=null),lo0=Math.min(...all),hi0=Math.max(...all),pad=(hi0-lo0)*.12||1,lo=o.ymin??lo0-pad,hi=hi0+pad;
  const xv=o.numeric?o.xs:o.xs.map((_,i)=>i),xl=Math.min(...xv),xh=Math.max(...xv),
    X=v=>L+(xv.length>1?(v-xl)/(xh-xl):.5)*(w-L-Rm-(o.numeric?0:30))+(o.numeric?0:15),Y=v=>T+(hi-v)/(hi-lo)*(h-T-B);
  const ax=el("g",{class:"ax"},svg);niceTicks(lo,hi,5).forEach(t=>{if(t<lo||t>hi)return;el("line",{x1:L,x2:w-Rm,y1:Y(t),y2:Y(t)},ax);el("text",{x:L-8,y:Y(t)+4,"text-anchor":"end"},ax).textContent=t});
  o.xs.forEach((v,i)=>{const px=X(xv[i]);el("text",{x:px,y:h-B+18,"text-anchor":"middle"},ax).textContent=o.numeric?v:v});
  if(lo<0&&hi>0)el("line",{x1:L,x2:w-Rm,y1:Y(0),y2:Y(0),stroke:COL.ink,"stroke-width":1.2},svg);
  if(o.xlabel)el("text",{x:(L+w-Rm)/2,y:h-6,"text-anchor":"middle",style:"font-size:11.5px;fill:#6b6b6b;font-weight:600"},svg).textContent=o.xlabel;
  if(o.ylabel)el("text",{x:L-8,y:T-2,"text-anchor":"start",style:"font-size:11px;fill:#a3a3a3"},svg).textContent=o.ylabel;
  o.series.forEach((s,k)=>{const pts=s.vals.map((v,i)=>v==null?null:[X(xv[i]),Y(v)]).filter(Boolean);if(!pts.length)return;
    const d=pts.map((p,i)=>(i?"L":"M")+p[0].toFixed(1)+" "+p[1].toFixed(1)).join("");
    const p=el("path",{d,fill:"none",stroke:s.color,"stroke-width":s.dash?2.2:3.2,"stroke-linecap":"round","stroke-linejoin":"round","stroke-dasharray":s.dash||"none"},svg);
    if(!s.dash){const len=p.getTotalLength();p.style.strokeDasharray=len;p.style.strokeDashoffset=len;p.style.transition=`stroke-dashoffset 1.2s cubic-bezier(.2,.8,.2,1) ${k*120}ms`;requestAnimationFrame(()=>requestAnimationFrame(()=>p.style.strokeDashoffset=0))}
    if(pts.length<=12)pts.forEach(q=>el("circle",{cx:q[0],cy:q[1],r:3.6,fill:"#fff",stroke:s.color,"stroke-width":2},svg))});
  const cl=el("line",{y1:T,y2:h-B,stroke:COL.ink,"stroke-width":1,"stroke-dasharray":"3 4",opacity:0},svg),ov=el("rect",{x:L,y:T,width:w-L-Rm,height:h-T-B,fill:"transparent"},svg);
  ov.onmousemove=e=>{const r=svg.getBoundingClientRect(),mx=(e.clientX-r.left)*(w/r.width);let bi=0,bd=1e9;xv.forEach((v,i)=>{const d=Math.abs(X(v)-mx);if(d<bd){bd=d;bi=i}});
    cl.setAttribute("x1",X(xv[bi]));cl.setAttribute("x2",X(xv[bi]));cl.setAttribute("opacity",.7);
    showTip(e,`<b>${o.xs[bi]}${o.xunit||""}</b><br>`+o.series.map(s=>s.vals[bi]==null?"":`<span style="color:${s.color==COL.ink?"#fff":s.color}">●</span> ${s.label}: <b>${s.vals[bi].toFixed(o.dec??1)}</b>`).join("<br>"))};
  ov.onmouseleave=()=>{cl.setAttribute("opacity",0);hideTip()}}

function dumbbell(id,rows,o={}){
  const {svg,w,h}=mount(id),L=Math.min(130,w*.26),Rm=70,T=20,B=28,all=rows.flatMap(r=>[r.a.v,r.b.v]),lo0=Math.min(...all),hi0=Math.max(...all),pad=(hi0-lo0)*.25||.5,lo=Math.min(lo0-pad,o.zero?0:lo0-pad),hi=hi0+pad;
  const X=v=>L+(v-lo)/(hi-lo)*(w-L-Rm),rh=(h-T-B)/rows.length;
  const ax=el("g",{class:"ax"},svg);niceTicks(lo,hi,6).forEach(t=>{if(t<lo||t>hi)return;el("line",{x1:X(t),x2:X(t),y1:T,y2:h-B},ax);el("text",{x:X(t),y:h-8,"text-anchor":"middle"},ax).textContent=(o.tick||(v=>v))(+t.toFixed(3))});
  rows.forEach((r,i)=>{const y=T+i*rh+rh/2,g=el("g",{},svg);el("text",{x:L-14,y:y+4,"text-anchor":"end",class:"lbl"},g).textContent=r.label;
    const ln=el("line",{x1:X(r.a.v),x2:X(r.a.v),y1:y,y2:y,stroke:COL.ink,"stroke-width":5,"stroke-linecap":"round",style:`transition:all 1s cubic-bezier(.2,.8,.2,1) ${i*120}ms`},g);
    const dot=(p,k)=>{const c=el("circle",{cx:X(r.a.v),cy:y,r:13,fill:p.color,stroke:COL.ink,"stroke-width":2,style:`transition:all 1s cubic-bezier(.2,.8,.2,1) ${i*120+k*90}ms;cursor:pointer`},g);
      c.onmousemove=e=>showTip(e,`<b>${r.label}</b><br>${p.name}: <b>${(o.fmt||f2)(p.v)}</b>${p.ci?` ± ${p.ci.toFixed(2)}`:""}`);c.onmouseleave=hideTip;
      requestAnimationFrame(()=>requestAnimationFrame(()=>c.setAttribute("cx",X(p.v))));return c};
    dot(r.a,0);dot(r.b,1);requestAnimationFrame(()=>requestAnimationFrame(()=>{ln.setAttribute("x1",X(r.a.v));ln.setAttribute("x2",X(r.b.v))}));
    const d=r.b.v-r.a.v;el("text",{x:Math.max(X(r.a.v),X(r.b.v))+22,y:y+4,class:"lbl num",style:`fill:${d>=0?COL.purple:COL.blue}`},g).textContent=(d>=0?"+":"−")+Math.abs(d).toFixed(o.dec??2)})}

function colsGrouped(id,o){
  const {svg,w,h}=mount(id),L=40,Rm=10,T=24,B=48,vals=o.series.flatMap(s=>s.vals),lo=Math.min(0,...vals),hi=Math.max(...vals)*1.12,
    Y=v=>T+(hi-v)/(hi-lo)*(h-T-B),gw=(w-L-Rm)/o.groups.length,bw=Math.min(46,gw*.8/o.series.length);
  const ax=el("g",{class:"ax"},svg);niceTicks(lo,hi,5).forEach(t=>{if(t<lo||t>hi)return;el("line",{x1:L,x2:w-Rm,y1:Y(t),y2:Y(t)},ax);el("text",{x:L-8,y:Y(t)+4,"text-anchor":"end"},ax).textContent=t});
  el("text",{x:L-8,y:12,"text-anchor":"start",style:"font-size:11px;fill:#a3a3a3"},svg).textContent="dB";
  o.groups.forEach((gname,gi)=>{const cx=L+gi*gw+gw/2;const t=el("text",{x:cx,y:h-24,"text-anchor":"middle",class:"lbl"},svg);gname.split("|").forEach((ln,k)=>{const s=el("tspan",{x:cx,dy:k?15:0},t);s.textContent=ln});
    o.series.forEach((s,si)=>{const bx=cx-(o.series.length*bw+(o.series.length-1)*5)/2+si*(bw+5),v=s.vals[gi],y0=Y(0),y1=Y(v);
      const r=el("rect",{x:bx,y:y0,width:bw,height:0,rx:5,fill:s.color,stroke:COL.ink,"stroke-width":1.5,style:`transition:all .9s cubic-bezier(.2,.8,.2,1) ${gi*110+si*60}ms;cursor:pointer`},svg);
      requestAnimationFrame(()=>requestAnimationFrame(()=>{r.setAttribute("y",Math.min(y0,y1));r.setAttribute("height",Math.max(2,Math.abs(y1-y0)))}));
      el("text",{x:bx+bw/2,y:Math.min(y0,y1)-6,"text-anchor":"middle",class:"lbl num",style:"font-size:11px"},svg).textContent=v.toFixed(1);
      r.onmousemove=e=>showTip(e,`<b>${s.label}</b><br>${gname.replace("|"," ")}: <b>${v.toFixed(2)} dB</b>`);r.onmouseleave=hideTip})})}

/* ---------- spectrogram cards + audio ---------- */
const CM=[[255,255,255],[100,168,176],[35,85,204],[11,11,12]];
function cmap(u){u=Math.max(0,Math.min(1,u))*(CM.length-1);const i=Math.min(CM.length-2,Math.floor(u)),f=u-i;return CM[i].map((c,k)=>Math.round(c+(CM[i+1][k]-c)*f))}
let playing=null;
function buildSpecs(){
  const ex=R.example,box=$("#specs");if(box.dataset.built)return;box.dataset.built=1;box.innerHTML="";
  const vmax=Math.max(...ex.spec.map(s=>Math.max(...s.flat()))),vmin=vmax-62;
  ex.spec.forEach((S,k)=>{const d=document.createElement("div");d.className="spec";d.innerHTML=`<canvas></canvas><div class="cap"><span>${ex.labels[k]}</span><span class="play">▶</span></div>`;box.appendChild(d);
    const c=$("canvas",d),F=S.length,Nn=S[0].length;c.width=Nn;c.height=F;const x=c.getContext("2d"),im=x.createImageData(Nn,F);
    for(let r=0;r<F;r++)for(let q=0;q<Nn;q++){const col=cmap((S[F-1-r][q]-vmin)/(vmax-vmin)),o=(r*Nn+q)*4;im.data[o]=col[0];im.data[o+1]=col[1];im.data[o+2]=col[2];im.data[o+3]=255}x.putImageData(im,0,0);
    const a=new Audio("data:audio/wav;base64,"+ex.audio[k]);let raf;
    const stop=()=>{a.pause();a.currentTime=0;d.classList.remove("playing");$(".play",d).textContent="▶";cancelAnimationFrame(raf);head(0)};
    const head=p=>{x.putImageData(im,0,0);if(p>0){x.fillStyle=COL.yellow;x.fillRect(Math.floor(p*Nn),0,1,F)}};
    const tick=()=>{head(a.currentTime/(a.duration||3));raf=requestAnimationFrame(tick)};
    a.onended=stop;d.onclick=()=>{if(playing&&playing!==stop)playing();if(d.classList.contains("playing")){stop();playing=null;return}
      a.play();d.classList.add("playing");$(".play",d).textContent="❚❚";playing=stop;tick()}})}

/* ---------- slide logic ---------- */
const M=R.meta,U=SYS.unproc;
const pr=(id,key)=>SYS[id][key].mean;
function initStatic(){
  hero();
  $("#m-stft").textContent=`${Math.round(M.win_ms)} ms window · ${Math.round(1000*M.hop/16000)} ms hop`;
  $("#m-train").innerHTML=`${M.n_train} training mixtures, re-mixed on the fly (wearer 5 to 30 dB above the partners, noise −6 to 12 dB), ${M.steps} steps of ${M.batch}. Loss: SI-SDR plus compressed-magnitude L1. Best checkpoint chosen on ${M.n_val} validation mixtures; ${M.n_test} test mixtures come from ${M.spk.test} unseen speakers and unseen noise segments.`;
  chips("m-chips",[1,2,4].map(m=>({label:m+(m>1?" mics":" mic"),m})),it=>{const k=1+3*(it.m-1);$("#m-maps").textContent=k;$("#m-dim").textContent=k*257;$("#m-par").textContent=(SYS["m"+it.m+"_causal"].params/1e6).toFixed(2)});
  // takeaways
  const s1=SYS.m1_causal,s4=SYS.m4_causal,c4=SYS.m4_causal,n4=SYS.m4_noncausal,d=s4.sisdri.mean-s1.sisdri.mean,dl=n4.sisdri.mean-c4.sisdri.mean;
  const ov=R.sweeps.ovr,i4=ov.series.m4_causal,wi=i4.indexOf(Math.min(...i4)),gap=ov.series.m4_causal.map((v,i)=>v-ov.series.m1_causal[i]);
  const T=[["RQ1",COL.blue,"#fff",`Four microphones change the SI-SDR improvement by <b>${sgn(d)} dB</b> versus one (${f1(s4.sisdri.mean)} vs ${f1(s1.sisdri.mean)} dB), and wearer-voice removal from ${f1(s1.own_sup.mean)} to ${f1(s4.own_sup.mean)} dB.`],
   ["RQ2",COL.purple,"#fff",`Letting the 4-mic model see the whole segment moves SI-SDR improvement by <b>${sgn(dl)} dB</b>, against a 32 ms causal system that can run in real time.`],
   ["RQ3",COL.yellow,COL.ink,`Across wearer loudness the 4-mic model scores between <b>${f1(Math.min(...i4))}</b> and <b>${f1(Math.max(...i4))} dB</b> improvement (hardest at ${ov.x[wi]} dB); its gain over one mic ranges from ${sgn(Math.min(...gap))} to ${sgn(Math.max(...gap))} dB.`]];
  $("#take").innerHTML=T.map((t,i)=>`<div class="card rq r" style="--i:${i+2};padding:14px 16px"><div class="tag" style="background:${t[1]};color:${t[2]};width:46px;height:46px">${t[0]}</div><p class="small" style="color:#222;font-size:13.5px">${t[3]}</p></div>`).join("")}

hooks[0]=()=>{};
hooks[2]=()=>{const s=R.step1;[["utts",0],["speakers",0],["hours",2]].forEach(([k,dec])=>countUp($(`[data-count=${k}]`),s[k],dec));
  $("#p-ovr").textContent=s.pilot.ovr.toFixed(1);$("#p-stoi").textContent=s.pilot.stoi.toFixed(2);$("#p-pesq").textContent=s.pilot.pesq.toFixed(2);
  const rows=[...s.demand].sort((a,b)=>a.level-b.level).map((d,i)=>({label:d.name,v:d.level+60,text:d.level.toFixed(1)+" dBFS",color:[COL.teal,COL.purple,COL.yellow,COL.blue][i],
    tip:`<b>${d.name}</b><br>level ${d.level.toFixed(1)} dBFS<br>inter-channel correlation <b>${d.corr.toFixed(2)}</b><br>pilot STOI before processing: <b>${s.pilot.stoi_env[d.env].toFixed(2)}</b>`}));
  barsH("c-noise",rows,{min:0,max:30,tick:v=>(v-60)+""})};
const RQ1M=[["sisdri","SI-SDR improvement (dB)",f1,"dB"],["stoi","STOI",f3,""],["pesq","PESQ (wideband)",f2,""],["own_sup","Wearer voice removed (dB)",f1,"dB"]];
hooks[4]=()=>{const ids=["unproc","m1_causal","m2_causal","m4_causal","oracle"],s1=SYS.m1_causal,s2=SYS.m2_causal,s4=SYS.m4_causal,or=SYS.oracle,d=s4.sisdr.mean-s1.sisdr.mean;
  $("#rq1-h").innerHTML=Math.abs(d)<.5?`More microphones <em>barely</em> move it.`:d>0?`Four microphones add <em>${d.toFixed(1)} dB</em>.`:`More microphones did <em>not</em> help here.`;
  $("#rq1-t").innerHTML=`SI-SDR goes from <b>${f1(U.sisdr.mean)}</b> dB (unprocessed) to <b>${f1(s1.sisdr.mean)}</b> (1 mic), <b>${f1(s2.sisdr.mean)}</b> (2 mics) and <b>${f1(s4.sisdr.mean)}</b> dB (4 mics). An oracle mask reaches ${f1(or.sisdr.mean)} dB, so ${f1(or.sisdr.mean-s4.sisdr.mean)} dB of headroom remain.`;
  legend("rq1-leg",ids);
  chips("rq1-chips",RQ1M.map(m=>({label:m[1].replace(" (dB)","").replace(" (wideband)",""),m})),it=>{const m=it.m;
    barsH("c-rq1",ids.map(i=>({label:R.labels[i].replace(" (upper bound)",""),v:SYS[i][m[0]].mean,ci:SYS[i][m[0]].ci,color:SC[i]})),{fmt:m[2],xlabel:m[3]})});
  const cs=["m1_causal","m2_causal","m4_causal"];
  lineChart("c-curve",{xs:R.curves.m1_causal.steps,numeric:true,xlabel:"training step",ylabel:"dB",series:cs.map(k=>({id:k,label:R.labels[k],color:SC[k],vals:R.curves[k].val}))})};
hooks[5]=()=>{const mk=(m,k)=>({a:{v:SYS["m"+m+"_causal"][k].mean,ci:SYS["m"+m+"_causal"][k].ci,color:COL.blue,name:"Causal"},b:{v:SYS["m"+m+"_noncausal"][k].mean,ci:SYS["m"+m+"_noncausal"][k].ci,color:COL.purple,name:"Non-causal"}});
  const dl=[1,4].map(m=>SYS["m"+m+"_noncausal"].sisdri.mean-SYS["m"+m+"_causal"].sisdri.mean),avg=(dl[0]+dl[1])/2,pr_=SYS.m4_noncausal.params/SYS.m4_causal.params;
  $("#rq2-h").innerHTML=Math.abs(avg)<.5?`Low latency is <em>nearly free</em>.`:avg>0?`Seeing everything buys <em>${avg.toFixed(1)} dB</em>.`:`Look-ahead does <em>not</em> pay off here.`;
  $("#rq2-t").innerHTML=`With the whole ${M.seg_s} s segment available, SI-SDR improvement changes by <b>${sgn(dl[0])} dB</b> (1 mic) and <b>${sgn(dl[1])} dB</b> (4 mics), with about ${pr_.toFixed(1)}× the parameters. The causal model only needs the current 32 ms window.`;
  $("#rq2-nc").innerHTML=`${M.seg_s}<span style="font-size:.45em"> s</span>`;$("#rq2-sp").textContent=SYS.m4_causal.ms_per_s.toFixed(0);
  chips("rq2-chips",RQ1M.map(m=>({label:m[1].replace(" (dB)","").replace(" (wideband)",""),m})),it=>{const m=it.m;
    dumbbell("c-rq2",[{label:"1 microphone",...mk(1,m[0])},{label:"4 microphones",...mk(4,m[0])}],{fmt:m[2],dec:m[0]==="stoi"?3:m[0]==="pesq"?2:1})})};
hooks[6]=()=>{const ids=["m1_causal","m2_causal","m4_causal","oracle"],dn=R.step1.demand,nm=Object.fromEntries(dn.map(d=>[d.env,d.name]));
  const T=[{label:"Wearer loudness",key:"ovr",x:R.sweeps.ovr.x,ser:R.sweeps.ovr.series,xl:"wearer level above partners (dB)",unit:" dB"},
   {label:"Noise level",key:"snr",x:R.sweeps.snr.x,ser:R.sweeps.snr.series,xl:"partners-to-noise ratio (dB)",unit:" dB"},
   {label:"Reverberation",key:"rt",x:R.by_rt.names,ser:Object.fromEntries(ids.map(k=>[k,R.by_rt.series[k].sisdri])),xl:"nominal RT60",num:false},
   {label:"Noise type",key:"env",x:R.by_env.names.map(e=>nm[e]||e),ser:Object.fromEntries(ids.map(k=>[k,R.by_env.series[k].sisdri])),xl:"noise scene",num:false}];
  legend("rq3-leg",ids);
  chips("rq3-chips",T.map(t=>({label:t.label,t})),it=>{const t=it.t,a=t.ser.m4_causal,b=t.ser.m1_causal,g=a.map((v,i)=>v-b[i]),wi=a.indexOf(Math.min(...a)),gi=g.indexOf(Math.max(...g));
    $("#rq3-h").innerHTML=`Stress test: <em>${t.label.toLowerCase()}</em>.`;
    $("#rq3-t").innerHTML=`The 4-mic model's gain over a single mic runs from <b>${sgn(Math.min(...g))}</b> to <b>${sgn(Math.max(...g))} dB</b> (largest at ${t.x[gi]}${t.unit||""}). Its worst case is <b>${f1(a[wi])} dB</b> improvement at ${t.x[wi]}${t.unit||""}.`;
    $("#rq3-cap").innerHTML=`<b style="color:var(--ink)">SI-SDR improvement (dB)</b> · hover the plot`;
    lineChart("c-rq3",{xs:t.x,numeric:t.num!==false,xlabel:t.xl,xunit:t.unit||"",series:ids.map(k=>({id:k,label:R.labels[k],color:SC[k],vals:t.ser[k],dash:k==="oracle"?"6 5":null}))})})};
hooks[7]=()=>{const ids=["m1_causal","m2_causal","m4_causal","oracle"],s4=SYS.m4_causal,s1=SYS.m1_causal;
  $("#an-h").innerHTML=`Where the dB go: <em>${f1(s4.own_sup.mean)} dB</em> off the wearer.`;
  $("#an-t").innerHTML=`The 4-mic model removes <b>${f1(s4.own_sup.mean)} dB</b> of the wearer's voice (1 mic: ${f1(s1.own_sup.mean)} dB) and <b>${f1(s4.noise_sup.mean)} dB</b> of the noise, while attenuating the partners' speech by <b>${f1(s4.tgt_loss.mean)} dB</b>. The wearer starts roughly ${R.step1.pilot.ovr.toFixed(0)} dB louder than the partners.`;
  legend("an-leg",ids);
  colsGrouped("c-an",{groups:["Wearer removed","Noise removed","Partner speech|lost (lower is better)"],series:ids.map(k=>({label:R.labels[k],color:SC[k],vals:[SYS[k].own_sup.mean,SYS[k].noise_sup.mean,SYS[k].tgt_loss.mean]}))})};
hooks[8]=()=>{const i=R.example.info,s=i.sisdri;buildSpecs();
  $("#ex-info").innerHTML=`Scene: <b>${(R.step1.demand.find(d=>d.env===i.env)||{name:i.env}).name}</b> noise · nominal RT60 ${i.rt60.toFixed(2)} s · wearer ${i.ovr.toFixed(0)} dB above the partners · partners-to-noise ratio ${sgn(i.snr)} dB. SI-SDR improvement on this clip: 1 mic <b>${f1(s.m1_causal)}</b> dB, 4 mics <b>${f1(s.m4_causal)}</b> dB, 4 mics non-causal <b>${f1(s.m4_noncausal)}</b> dB.`};
let rz;addEventListener("resize",()=>{clearTimeout(rz);rz=setTimeout(()=>{const f=hooks[cur];if(f&&cur>=4&&cur<=7)f()},250)});
initStatic();go(0);

</script></body></html>"""

html = TEMPLATE.replace("/*__RESULTS__*/null", json.dumps(R, separators=(",", ":")))
Path("echi_proxy_presentation.html").write_text(html, encoding="utf-8")
print(f"written echi_proxy_presentation.html ({len(html)/1e6:.2f} MB) | total {(time.time()-T_START)/60:.1f} min")
try:
    from google.colab import files; files.download("echi_proxy_presentation.html")
except Exception: pass


written echi_proxy_presentation.html (0.81 MB) | total 29.1 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>